# Aggregated multi-setup analysis

For a fixed set of architectures (one hyperparameter configuration per architecture), this
notebook combines, into a single tidy DataFrame (one row per setup), the four analyses that
were previously scattered across separate notebooks:

- **R² per SSP scenario** — same computation as `CMIP_analysis_variable_lamda_fast.ipynb`
  (built from each setup's `quality_df.pkl`).
- **Climate-classifier accuracy** (climate-invariance proxy) — same protocol (PCA 16D + MLP)
  as `CMIP_analysis_latent.ipynb` Part 2.
- **Alignment between historical and ssp585** (sliced-Wasserstein distance) — same protocol
  as `CMIP_analysis_latent.ipynb` Part 3.
- **Physical-organization ratio** over random latent directions — same protocol as
  `CMIP_analysis_latent_ratio.ipynb`.

The final DataFrame is saved to `reportGraph/` and returned automatically at the end of the
notebook.

## Setups compared

Seven architectures, one hyperparameter configuration each (values fixed in Part 0):

| Setup | quality_df dir | alignment method | latent dims used for classifier / alignment / ratio |
|---|---|---|---|
| Baseline No Align | Baseline_CERA_noalign | – | first 48 |
| CERA | CERA | swd | first 48 |
| CERA SWDN | CERA | swdn | first 48 |
| CERA adversarial | CERA | adversarial | first 48 |
| CERA full latent | CERA_full_latent | swd | all 64 |
| CERA seasonal | CERA_seasonal | swd | first 48 |
| Baseline ClimaX | Baseline_ClimaX | – | all 64 |

R² (Part 2) always uses the full prediction output stored in `quality_df.pkl` — it does not
depend on the latent slicing used for the other three metrics.

In [1]:
import numpy as np
import pandas as pd
import json
import pickle
from pathlib import Path
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import NearestNeighbors
from scipy.stats import wasserstein_distance
from IPython.display import display

## Part 0 — Configuration

In [ ]:
num_sample = 1000000
chosen_autoencoder_type = "CNN"   # choose between "MLP" and "CNN"
inv_alignment_method = "swd"      # base/default alignment method (used for output naming; overridden per-setup in Part 1)
variable = "pr"                   # variable to predict / excluded from the 15 raw input variables
val_fraction = 0.05
test_fraction = 0.15
cera_lambda_align = 0.0001
cera_lambda_pred = 0.01

evaluation_root = Path("/glade/work/tsalin/CMIP/model_evaluation")
latent_root = Path("/glade/work/tsalin/CMIP/latent_representations")
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v{num_sample}")
report_dir = Path("/glade/u/home/tsalin/CMIP/reportGraph")

for _dir in (evaluation_root, latent_root, precomputed_dir):
    if not _dir.exists():
        raise FileNotFoundError(f"Required directory not found: {_dir}")
report_dir.mkdir(parents=True, exist_ok=True)

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])          # ["historical", "ssp126", "ssp245", "ssp370", "ssp585"]
random_seed = int(run_cfg["random_seed"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
assert n_lat * n_lon == grid_points_per_patch
assert "ssp585" in climate_order and "historical" in climate_order

print(f"Climates: {climate_order}")
print(f"random_seed = {random_seed}")

## Part 1 — Setup registry

Each setup carries everything needed to locate its `quality_df.pkl` (for R², Part 2) and its
`latent_representations_df.pkl` (for the classifier, alignment and ratio metrics, Parts 4-7),
plus the number of leading latent dimensions to keep for those three metrics (`slice_dim=None`
means "keep all dimensions").

In [ ]:
def _cera_family_prefix(base, alignment_method, autoencoder_type):
    """Filename prefix shared by CERA / CERA SWDN / CERA adversarial / CERA full latent / CERA seasonal."""
    return (
        f"{base}_ns{num_sample}_{autoencoder_type}_{alignment_method}_{variable}_"
        f"{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_"
    )


SETUPS = {
    "Baseline No Align": {
        "quality_dir": "Baseline_CERA_noalign",
        "quality_prefix": f"baseline_CERA_noalign_ns{num_sample}_{chosen_autoencoder_type}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_pred}_",
        "latent_dir": "latent_representations_exp_5",
        "latent_prefix": f"baseline_CERA_noalign_ns{num_sample}_{chosen_autoencoder_type}_{variable}_{val_fraction}_{test_fraction}_{cera_lambda_pred}_",
        "slice_dim": 48,
    },
    "CERA": {
        "quality_dir": "CERA",
        "quality_prefix": _cera_family_prefix("cera", "swd", "CNN"),
        "latent_dir": "latent_representations_exp_5",
        "latent_prefix": _cera_family_prefix("cera", "swd", "CNN"),
        "slice_dim": 48,
    },
    "CERA MLP SWD": {
        "quality_dir": "CERA",
        "quality_prefix": _cera_family_prefix("cera", "swd", "MLP"),
        "latent_dir": "latent_representations_exp_5",
        "latent_prefix": _cera_family_prefix("cera", "swd", "MLP"),
        "slice_dim": 48,
    },
    "CERA full latent": {
        "quality_dir": "CERA_full_latent",
        "quality_prefix": _cera_family_prefix("cera_full_latent", "swd", "CNN"),
        "latent_dir": "latent_representations_exp_5",
        "latent_prefix": _cera_family_prefix("cera_full_latent", "swd", "CNN"),
        "slice_dim": None,
    },
    "Baseline ClimaX": {
        "quality_dir": "Baseline_ClimaX",
        "quality_prefix": f"baseline_ClimaX_ns{num_sample}_{variable}_{val_fraction}_{test_fraction}_",
        "latent_dir": "latent_representations_exp_5",
        "latent_prefix": f"baseline_ClimaX_ns{num_sample}_{variable}_{val_fraction}_{test_fraction}_",
        "slice_dim": None,
    },
}
"""

SETUPS = {
    "Baseline ClimaX": {
        "quality_dir": "Baseline_ClimaX",
        "quality_prefix": f"baseline_ClimaX_boosted_ns{num_sample}_{variable}_{val_fraction}_{test_fraction}_",
        "latent_dir": "latent_representations_exp_5",
        "latent_prefix": f"baseline_ClimaX_boosted_ns{num_sample}_{variable}_{val_fraction}_{test_fraction}_",
        "slice_dim": None,
    },
}
"""

for name, spec in SETUPS.items():
    spec["quality_path"] = evaluation_root / spec["quality_dir"] / f"{spec['quality_prefix']}quality_df.pkl"
    spec["latent_path"] = latent_root / spec["latent_dir"] / f"{spec['latent_prefix']}latent_representations_df.pkl"

for name, spec in SETUPS.items():
    print(f"{name:20s}  slice_dim={str(spec['slice_dim']):4s}  quality={spec['quality_path'].name}")
    print(f"{'':20s}  latent={spec['latent_path'].name}")

## Part 2 — R² per scenario (5 climates), from `quality_df.pkl`

In [ ]:
def _safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if y_true.size < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))


def _parse_variable_slices(value_names):
    """Return {var_name: slice} in column order, one entry per unique variable."""
    slices = []
    seen = {}
    for i, name in enumerate(value_names):
        var = str(name).split("@")[0]
        if var not in seen:
            seen[var] = i
            slices.append((var, i))
    result = {}
    for k, (var, start) in enumerate(slices):
        end = slices[k + 1][1] if k + 1 < len(slices) else len(value_names)
        result[var] = slice(start, end)
    return result


def compute_r2_by_climate(quality_path, variable):
    with open(quality_path, "rb") as fh:
        payload = pickle.load(fh)

    meta = pd.DataFrame(payload["meta_prediction"]).reset_index(drop=True)
    truth = np.asarray(payload["truth_prediction"])
    pred = np.asarray(payload["pred_prediction"])
    var_slice = _parse_variable_slices(payload["prediction_value_names"])[variable]

    r2_by_climate = {}
    for climate, idx in meta.groupby("scenario", sort=False).groups.items():
        idx = np.asarray(idx)
        t = truth[idx][:, var_slice].ravel()
        p = pred[idx][:, var_slice].ravel()
        r2_by_climate[str(climate)] = _safe_r2(t, p)
    return r2_by_climate


r2_by_setup = {}
for name, spec in SETUPS.items():
    r2_by_setup[name] = compute_r2_by_climate(spec["quality_path"], variable)
    print(f"{name:20s}  " + "  ".join(f"{c}={r2_by_setup[name].get(c, float('nan')):.3f}" for c in climate_order))

## Part 3 — Load latent representations (own latent per setup, sliced)

In [ ]:
def _load_latent_payload(file_path):
    if not file_path.exists():
        raise FileNotFoundError(f"Latent representations file not found: {file_path}")
    with open(file_path, "rb") as handle:
        payload = pickle.load(handle)
    latent_by_climate = {}
    metadata_by_climate_latent = {}
    for climate, entry in payload.items():
        latent_by_climate[climate] = np.asarray(entry["latent"])
        metadata_by_climate_latent[climate] = pd.DataFrame(entry["metadata"]).reset_index(drop=True)
    return latent_by_climate, metadata_by_climate_latent


def _sliced(X, slice_dim):
    return X if slice_dim is None else X[:, :slice_dim]


latent_by_setup = {}       # {setup_name: {climate: latent array, sliced}}
latent_meta_by_setup = {}  # {setup_name: {climate: metadata df}}

for name, spec in SETUPS.items():
    lat, meta = _load_latent_payload(spec["latent_path"])
    latent_by_setup[name] = {c: _sliced(lat[c], spec["slice_dim"]) for c in climate_order}
    latent_meta_by_setup[name] = meta
    dim = latent_by_setup[name]["historical"].shape[1]
    print(f"{name:20s}  latent_dim_used={dim}")

## Part 4 — Climate-invariance proxy: MLP classifier accuracy

Same protocol as `CMIP_analysis_latent.ipynb` Part 2 (MLP branch): PCA (16D, fit on train only)
followed by an `MLPClassifier` (2×64, ReLU, Adam, early stopping) trained to predict which of
the 5 climates a latent vector comes from, with balanced sample counts per climate. A single
70/30 train/test split is used (simplified from the original notebook's 70/15/15, since only
test accuracy is kept here). Lower accuracy = more climate-invariant
(random baseline = 1/5 = 0.20).

In [ ]:
CLASSIFIER_PCA_DIM = 16
CLASSIFIER_TRAIN_FRAC = 0.70
climate_label = {c: i for i, c in enumerate(climate_order)}


def compute_classifier_accuracy(latent_by_climate, seed):
    rng = np.random.default_rng(seed)
    split_indices = {}
    for climate in climate_order:
        N = latent_by_climate[climate].shape[0]
        idx = rng.permutation(N)
        n_train = int(CLASSIFIER_TRAIN_FRAC * N)
        split_indices[climate] = {"train": idx[:n_train], "test": idx[n_train:]}

    n_train = min(len(split_indices[c]["train"]) for c in climate_order)
    n_test = min(len(split_indices[c]["test"]) for c in climate_order)

    def _stack(split_key, n_per_climate):
        X_parts, y_parts = [], []
        for climate in climate_order:
            idx = split_indices[climate][split_key][:n_per_climate]
            X_parts.append(latent_by_climate[climate][idx])
            y_parts.append(np.full(n_per_climate, climate_label[climate]))
        return np.vstack(X_parts), np.concatenate(y_parts)

    X_train_raw, y_train = _stack("train", n_train)
    X_test_raw, y_test = _stack("test", n_test)

    n_pca = min(CLASSIFIER_PCA_DIM, X_train_raw.shape[1], X_train_raw.shape[0] - 1)
    pca = PCA(n_components=n_pca, random_state=seed)
    X_train = pca.fit_transform(X_train_raw)
    X_test = pca.transform(X_test_raw)

    mlp = MLPClassifier(
        hidden_layer_sizes=(64, 64), activation="relu", solver="adam",
        max_iter=500, random_state=seed, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=20,
    )
    mlp.fit(X_train, y_train)
    return float(mlp.score(X_test, y_test))


classifier_accuracy_by_setup = {}
for name in SETUPS:
    classifier_accuracy_by_setup[name] = compute_classifier_accuracy(latent_by_setup[name], seed=random_seed)
    print(f"{name:20s}  test_accuracy={classifier_accuracy_by_setup[name]:.4f}")

print(f"\nRandom baseline: {1.0 / len(climate_order):.4f}")

## Part 5 — Alignment between historical and ssp585 (sliced-Wasserstein distance)

Latents are first re-centered/rescaled using the historical cloud's own centroid and RMS
dispersion (`z' = (z - mu_hist) / s_hist`), then the sliced-Wasserstein distance between the
normalised historical and ssp585 clouds is computed (200 random 1-D projections). Same
protocol as `CMIP_analysis_latent.ipynb` Part 3.

In [ ]:
SWD_N_PROJECTIONS = 200


def sliced_wasserstein(X, Y, n_projections, seed):
    rng = np.random.default_rng(seed)
    proj = rng.standard_normal((n_projections, X.shape[1]))
    proj /= np.linalg.norm(proj, axis=1, keepdims=True) + 1e-12
    return float(np.mean([wasserstein_distance(X @ p, Y @ p) for p in proj]))


def compute_alignment_swd(latent_by_climate, seed):
    X_hist = latent_by_climate["historical"]
    mu_hist = X_hist.mean(axis=0)
    s_hist = float(np.sqrt(np.mean(np.sum((X_hist - mu_hist) ** 2, axis=1))))
    X_hist_norm = (X_hist - mu_hist) / s_hist
    X_585_norm = (latent_by_climate["ssp585"] - mu_hist) / s_hist
    return sliced_wasserstein(X_hist_norm, X_585_norm, SWD_N_PROJECTIONS, seed)


alignment_swd_by_setup = {}
for name in SETUPS:
    alignment_swd_by_setup[name] = compute_alignment_swd(latent_by_setup[name], seed=random_seed)
    print(f"{name:20s}  swd(historical, ssp585)={alignment_swd_by_setup[name]:.4f}")

## Part 6 — Physical-organization ratio: shared raw data & anchor pool

The ratio (Part 7) needs, for every setup, a comparison against a common "trivial matching"
baseline built from the 15 raw input variables. This part loads that raw data once (shared
across all setups) and builds one fixed pool of 10 000 historical anchors + their
nearest-neighbour match under the RawData baseline — exactly as in
`CMIP_analysis_latent_ratio.ipynb`, computed once here instead of per setup.

This is the heaviest part of the notebook (reloads the full raw grid data).

In [ ]:
# ── Raw data (needed only for the 15 input variables, in standardized form) ───
features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}
metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

selected_variables_full = list(run_cfg["selected_variables"])
if variable not in selected_variables_full:
    raise ValueError(f"Variable '{variable}' was not found in run_config selected_variables.")

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch
variable_var_index = selected_variables_full.index(variable)
variable_col_start = variable_var_index * grid_points_per_patch
variable_col_end = variable_col_start + grid_points_per_patch

feature_mask_without_variable = np.ones(expected_dim_full, dtype=bool)
feature_mask_without_variable[variable_col_start:variable_col_end] = False

features_by_climate = {}
for c in climate_order:
    X_full = features_by_climate_full[c]
    if X_full.shape[1] != expected_dim_full:
        raise ValueError(f"Unexpected feature dimension for {c}: got {X_full.shape[1]}, expected {expected_dim_full}.")
    features_by_climate[c] = X_full[:, feature_mask_without_variable]

selected_variables = [v for v in selected_variables_full if v != variable]
n_input_variables = len(selected_variables)
del features_by_climate_full

print(f"Raw input variables (excluding '{variable}'): {selected_variables}")
for c in climate_order:
    print(f"  {c}: {features_by_climate[c].shape}")

In [ ]:
# ── Train/test splits (same convention as the AE training) + RAM opt for eval-only climates ──
def build_split_indices(data_by_climate, val_fraction, test_fraction, seed):
    split_indices = {}
    rng = np.random.default_rng(seed)
    for climate, X in data_by_climate.items():
        n = X.shape[0]
        indices = np.arange(n)
        rng.shuffle(indices)
        n_test = max(1, int(round(test_fraction * n)))
        n_val = max(1, int(round(val_fraction * n)))
        n_train = max(1, n - n_val - n_test)
        split_indices[climate] = {
            "train": indices[:n_train],
            "val": indices[n_train:n_train + n_val],
            "test": indices[n_train + n_val:],
        }
    return split_indices


ae_split_indices = build_split_indices(features_by_climate, val_fraction, test_fraction, random_seed)

_eval_only_climates = [c for c in climate_order if c != "historical"]
for c in _eval_only_climates:
    _idx = ae_split_indices[c]["test"]
    features_by_climate[c] = features_by_climate[c][_idx]
    metadata_by_climate[c] = metadata_by_climate[c].iloc[_idx].reset_index(drop=True)
    ae_split_indices[c] = {
        "train": np.array([], dtype=np.intp),
        "val": np.array([], dtype=np.intp),
        "test": np.arange(len(_idx), dtype=np.intp),
    }
print(f"RAM opt: {_eval_only_climates} truncated to their test split.")

In [ ]:
# ── Raw input variable standardization (mean/std fit on historical TRAIN only) ─
hist_train_idx = ae_split_indices["historical"]["train"]
X_hist_train_raw = np.asarray(features_by_climate["historical"][hist_train_idx], dtype=np.float32)

input_variable_means = np.zeros(n_input_variables, dtype=np.float64)
input_variable_stds = np.ones(n_input_variables, dtype=np.float64)

for var_idx, var_name in enumerate(selected_variables):
    cols_for_var = slice(var_idx * grid_points_per_patch, (var_idx + 1) * grid_points_per_patch)
    values = X_hist_train_raw[:, cols_for_var].reshape(-1)
    values = values[np.isfinite(values)]
    if var_name == "pr":
        values = np.log1p(values * 86400)
    mu, sigma = float(np.mean(values)), float(np.std(values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0
    input_variable_means[var_idx] = mu
    input_variable_stds[var_idx] = sigma


def standardize_input_variables(X_raw):
    X_raw = np.asarray(X_raw, dtype=np.float64)
    X_scaled = np.empty_like(X_raw, dtype=np.float32)
    for var_idx, var_name in enumerate(selected_variables):
        cols_for_var = slice(var_idx * grid_points_per_patch, (var_idx + 1) * grid_points_per_patch)
        values = X_raw[:, cols_for_var]
        if var_name == "pr":
            values = np.log1p(values * 86400)
        X_scaled[:, cols_for_var] = ((values - input_variable_means[var_idx]) / input_variable_stds[var_idx]).astype(np.float32)
    return X_scaled


raw_scaled_features_by_climate = {c: standardize_input_variables(features_by_climate[c]) for c in climate_order}
del X_hist_train_raw
print("Raw 15-variable standardization done.")

In [ ]:
# ── Shared anchor pool (10,000 historical test points) + RawData baseline pairing ──
def _sample_keys(meta_df):
    """Return an array of unique 'time|patch_id' string keys, one per row."""
    return (meta_df["time"].astype(str) + "|" + meta_df["patch_id"].astype(str)).values


ssp_climates = [c for c in climate_order if c != "historical"]

N_PAIRS_TARGET = 10000
PAIR_SEED = random_seed + 500
_pair_rng = np.random.default_rng(PAIR_SEED)

hist_test_idx = ae_split_indices["historical"]["test"]
hist_meta_test = metadata_by_climate["historical"].iloc[hist_test_idx].reset_index(drop=True)
hist_keys_all = _sample_keys(hist_meta_test)
n_avail_hist = len(hist_keys_all)
N_PAIRS = min(N_PAIRS_TARGET, n_avail_hist)

anchor_pos = _pair_rng.choice(n_avail_hist, size=N_PAIRS, replace=False)
anchor_keys = hist_keys_all[anchor_pos]

print(f"Anchor pool: {n_avail_hist} historical test points available -> using {N_PAIRS} anchors "
      f"(target was {N_PAIRS_TARGET}).")


def _patch_mean_summary(flat_block, n_vars, gpp):
    N = flat_block.shape[0]
    block = np.asarray(flat_block, dtype=np.float64).reshape(N, n_vars, gpp)
    return block.mean(axis=2).astype(np.float32)


def _raw_input_summary(climate):
    X_flat = raw_scaled_features_by_climate[climate]
    if climate == "historical":
        X_flat = X_flat[ae_split_indices["historical"]["test"]]
    return _patch_mean_summary(X_flat, n_input_variables, grid_points_per_patch)


raw_hist_summary_anchor = _raw_input_summary("historical")[anchor_pos]

raw_ssp_summary_parts, raw_ssp_climate_parts, raw_ssp_index_parts = [], [], []
for c in ssp_climates:
    S = _raw_input_summary(c)
    raw_ssp_summary_parts.append(S)
    raw_ssp_climate_parts.append(np.full(len(S), c))
    raw_ssp_index_parts.append(np.arange(len(S)))

raw_ssp_summary_pool = np.vstack(raw_ssp_summary_parts)
raw_ssp_pool_climate = np.concatenate(raw_ssp_climate_parts)
raw_ssp_pool_index = np.concatenate(raw_ssp_index_parts)

nn_raw = NearestNeighbors(n_neighbors=1).fit(raw_ssp_summary_pool)
_, raw_nn_pos = nn_raw.kneighbors(raw_hist_summary_anchor)
raw_nn_pos = raw_nn_pos[:, 0]

rawdata_pairing = (raw_ssp_pool_climate[raw_nn_pos], raw_ssp_pool_index[raw_nn_pos])
print(f"RawData baseline pairing built ({len(rawdata_pairing[0])} pairs).")

## Part 7 — Physical-organization ratio, per setup

For each setup: build its own nearest-neighbour pairing (the 10 000 shared anchors, matched
against their nearest neighbour in that setup's own sliced latent space), standardize on the
pooled historical+matched population, compute the population covariance `M_pop`, sample 300
random directions normalised so `Var(v · Z) = 1`, and compute
`ratio(v) = score_own(v) / score_RawData(v)` for each direction — same definition as
`CMIP_analysis_latent_ratio.ipynb`. `ratio_median` is the median over the 300 directions.

This is the most expensive part per setup (nearest-neighbour search + covariance fit).

In [ ]:
K_RANDOM_DIRECTIONS = 300
RANDOM_DIR_SEED = 42
RATIO_RIDGE_EPS = 1e-6


def build_own_latent_pairing(setup_name):
    lat = latent_by_setup[setup_name]
    meta = latent_meta_by_setup[setup_name]

    hist_lkeys = _sample_keys(meta["historical"])
    key_to_row = {k: i for i, k in enumerate(hist_lkeys)}
    missing = [k for k in anchor_keys if k not in key_to_row]
    if missing:
        raise ValueError(f"[{setup_name}] {len(missing)} anchor keys missing from its historical latent block.")
    hist_rows = np.array([key_to_row[k] for k in anchor_keys])
    Z_hist = lat["historical"][hist_rows]

    Z_ssp_parts, ssp_pool_climate_parts, ssp_pool_index_parts = [], [], []
    for c in ssp_climates:
        Zc = lat[c]
        keys_c = _sample_keys(meta[c])
        raw_key_to_row = {k: i for i, k in enumerate(_sample_keys(metadata_by_climate[c]))}
        idx_in_raw = np.array([raw_key_to_row[k] for k in keys_c])
        Z_ssp_parts.append(Zc)
        ssp_pool_climate_parts.append(np.full(len(Zc), c))
        ssp_pool_index_parts.append(idx_in_raw)

    Z_ssp_pool = np.vstack(Z_ssp_parts)
    ssp_pool_climate = np.concatenate(ssp_pool_climate_parts)
    ssp_pool_index = np.concatenate(ssp_pool_index_parts)

    nn = NearestNeighbors(n_neighbors=1).fit(Z_ssp_pool)
    _, nn_pos = nn.kneighbors(Z_hist)
    nn_pos = nn_pos[:, 0]

    return ssp_pool_climate[nn_pos], ssp_pool_index[nn_pos]


def get_Z_for_pairing(setup_name, target_climate, ssp_index):
    """Return (Z_hist, Z_ssp) — raw (unstandardised) latent vectors — for an arbitrary
    (target_climate, ssp_index) pairing, using setup_name's own (already sliced) latent."""
    lat = latent_by_setup[setup_name]
    meta = latent_meta_by_setup[setup_name]

    hist_lkeys = _sample_keys(meta["historical"])
    key_to_row = {k: i for i, k in enumerate(hist_lkeys)}
    hist_rows = np.array([key_to_row[k] for k in anchor_keys])
    Z_hist = lat["historical"][hist_rows]

    Z_ssp = np.empty_like(Z_hist)
    for c in np.unique(target_climate):
        mask = target_climate == c
        raw_key_to_row = {k: i for i, k in enumerate(_sample_keys(metadata_by_climate[c]))}
        keys_c = _sample_keys(meta[c])
        raw_row_to_latent_row = {raw_key_to_row[k]: i for i, k in enumerate(keys_c)}
        latent_rows = np.array([raw_row_to_latent_row[i] for i in ssp_index[mask]])
        Z_ssp[mask] = lat[c][latent_rows]

    return Z_hist, Z_ssp


def sample_random_Mpop_unit_directions(M_pop, dim, k, seed):
    """K random directions, each rescaled so v.T @ M_pop @ v = 1."""
    rng = np.random.default_rng(seed)
    raw = rng.normal(size=(k, dim))
    quad = np.einsum('ij,ij->i', raw, raw @ M_pop)
    return raw / np.sqrt(quad)[:, None]


def score_from_g(gh, gs):
    l_pair = np.mean((gh - gs) ** 2)
    var_g = np.var(np.concatenate([gh, gs]))
    return var_g / (l_pair + 1e-9)


def compute_ratio_median(setup_name):
    own_pairing = build_own_latent_pairing(setup_name)
    Z_hist, Z_ssp_own = get_Z_for_pairing(setup_name, *own_pairing)
    dim = Z_hist.shape[1]

    pool = np.vstack([Z_hist, Z_ssp_own])
    mean = pool.mean(axis=0)
    std = pool.std(axis=0)
    std_safe = np.where(std > 0, std, 1.0)

    Zh_own = (Z_hist - mean) / std_safe
    Zs_own = (Z_ssp_own - mean) / std_safe
    M_pop = np.cov(np.vstack([Zh_own, Zs_own]), rowvar=False) + RATIO_RIDGE_EPS * np.eye(dim)

    # Zh for the RawData pairing uses the exact same anchors -> identical to Zh_own, only Z_ssp differs.
    _, Zs_raw_unscaled = get_Z_for_pairing(setup_name, *rawdata_pairing)
    Zs_raw = (Zs_raw_unscaled - mean) / std_safe

    V = sample_random_Mpop_unit_directions(M_pop, dim, K_RANDOM_DIRECTIONS, seed=RANDOM_DIR_SEED)

    Gh = Zh_own @ V.T
    Gs_own = Zs_own @ V.T
    Gs_raw = Zs_raw @ V.T

    ratios = np.empty(K_RANDOM_DIRECTIONS)
    for k in range(K_RANDOM_DIRECTIONS):
        s_own = score_from_g(Gh[:, k], Gs_own[:, k])
        s_raw = score_from_g(Gh[:, k], Gs_raw[:, k])
        ratios[k] = s_own / s_raw

    return float(np.median(ratios))


ratio_median_by_setup = {}
for name in SETUPS:
    ratio_median_by_setup[name] = compute_ratio_median(name)
    print(f"{name:20s}  ratio_median={ratio_median_by_setup[name]:.3f}")

## Part 8 — Assemble final DataFrame and save

One row per setup, five columns: `setup`, `r2_by_climate` (dict of the 5 per-climate R²),
`classifier_accuracy_mlp`, `alignment_swd_hist_ssp585`, `ratio_median`. Saved to `reportGraph/`
under a name that encodes the hyperparameter combination in the same order as the CERA
`quality_df.pkl` / `latent_representations_df.pkl` filenames.

In [ ]:
rows = []
for name in SETUPS:
    rows.append({
        "setup": name,
        "r2_by_climate": r2_by_setup[name],
        "classifier_accuracy_mlp": classifier_accuracy_by_setup[name],
        "alignment_swd_hist_ssp585": alignment_swd_by_setup[name],
        "ratio_median": ratio_median_by_setup[name],
    })

agregate_df = pd.DataFrame(rows)

_output_name = (
    f"agregate_v2_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_{variable}_"
    f"{val_fraction}_{test_fraction}_{cera_lambda_align}_{cera_lambda_pred}_df.pkl"
)
_output_path = report_dir / _output_name
agregate_df.to_pickle(_output_path)
print(f"Saved: {_output_path}")

display(agregate_df)

In [ ]:
agregate_df